# Fase 6 — Avaliação

**Objetivo da fase:** medir o modelo ajustado no conjunto de **teste**, comparar com o baseline da Fase 1 nas mesmas métricas e registrar o resultado e a causa provável dos erros.

## 6.1 Preparar o ambiente

### 6.1.1 Configurar o Google Colab

Abrir o link do notebook e ir em *Arquivo → Salvar uma cópia no Drive*. A partir daqui, trabalhar sempre na cópia.

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecionar **GPU T4** e **Versão 2026.10**, a mesma da Fase 1. Usar um **runtime novo**, sem nada da Fase 5 na memória: o adaptador vem do Drive e o teste vem do Hub. Isso também confere que tudo o que a avaliação precisa foi salvo.

O secret `HF_TOKEN` criado na Fase 1 já fica disponível para todos os notebooks da conta. No painel lateral, ícone de chave, ligar *Acesso ao notebook* para o `HF_TOKEN` e rodar a célula abaixo para verificar a configuração.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))

conta = whoami()
print(f"Conta:         {conta['name']}")
print(f"Tipo de token: {conta['auth']['accessToken']['role']}")
# Esperado: o nome da conta e "write"

### 6.1.2 Instalar as bibliotecas

Instalar as bibliotecas através das linhas de comando abaixo. O `peft` carrega o adaptador sobre o modelo base e o `scikit-learn` calcula as métricas:

In [ ]:
# Versões fixadas para o runtime 2026.10 do Colab (ver README)
!pip install -q transformers==5.22.0 datasets==4.8.5 peft==0.21.2 accelerate==1.15.0 huggingface_hub==1.33.0 scikit-learn==1.6.1

## 6.2 Carregar o adaptador e o teste

### 6.2.1 Carregar o adaptador do Drive

Montar o Google Drive, ler a configuração da Fase 4 e carregar o modelo base com o adaptador salvo na Fase 5, do mesmo jeito que na seção 5.5.2.

Ao rodar a célula, o Colab pede permissão para acessar o Google Drive da conta:

In [ ]:
import json
import torch
from google.colab import drive
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

drive.mount("/content/drive")

PROJETO_DIR = "/content/drive/MyDrive/fine-tuning-soc"
ADAPTER_DIR = f"{PROJETO_DIR}/soc-triage-qwen2.5-0.5b-lora"

with open(f"{PROJETO_DIR}/config_treino.json") as f:
    config_treino = json.load(f)

MODEL_ID = config_treino["model_id"]

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print(f"Modelo base: {MODEL_ID}")
print(f"Adaptador:   {ADAPTER_DIR}")
print(f"Memória da GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Um único modelo serve para os dois lados da comparação. Com o adaptador ligado, é o modelo ajustado. Dentro de `model.disable_adapter()`, é exatamente o modelo base da Fase 1, sem nenhuma correção do LoRA. Assim o baseline não precisa de uma segunda cópia do modelo na GPU.

### 6.2.2 Carregar o conjunto de teste

Carregar o split de **teste** dos dois datasets publicados:

- **`alertas-soc-sft` (Fase 3):** traz o prompt do modelo ajustado, com o mesmo `SYSTEM` usado no treino.
- **`alertas-soc-splits` (Fase 2):** traz o `alerta`, o `contexto` e a `triagem` separados, que o prompt few-shot do baseline e a análise dos erros usam.

Os dois foram gerados na mesma ordem, então o exemplo `i` é o mesmo nos dois. A célula confere isso antes de seguir:

In [ ]:
from datasets import load_dataset

SPLITS_ID = f"{conta['name']}/alertas-soc-splits"

teste_sft = load_dataset(config_treino["sft_id"], split="test")
teste = load_dataset(SPLITS_ID, split="test")

assert len(teste_sft) == len(teste), "os dois testes têm tamanhos diferentes"
for ex_sft, ex in zip(teste_sft, teste):
    assert ex_sft["prompt"][1]["content"] == f"{ex['alerta']}\n{ex['contexto']}", "os dois testes não estão alinhados"

print(f"Exemplos de teste: {len(teste)}")
print(f"Templates:         {len(set(teste['template_id']))}")
# Esperado: 221 exemplos

Ver quantos exemplos de cada classe o teste tem. O número de críticos é o denominador do recall de "crítica", a métrica de negócio da Fase 1:

In [ ]:
import pandas as pd

reais = teste["triagem"]
ORDEM = ["critica", "alta", "media", "baixa"]

print(pd.Series([r["severidade"] for r in reais]).value_counts().reindex(ORDEM))
print()
print(pd.Series([r["veredito"] for r in reais]).value_counts())

O teste nunca foi usado para decidir nada nas fases anteriores: o treino só viu o `train`, e a Fase 5 acompanhou o `validation`. Além disso, a divisão da Fase 2 foi por template, então os padrões de alerta do teste são **novos** para o modelo. A métrica mede se ele aprendeu a política, e não se decorou exemplos.

## 6.3 Gerar as triagens

### 6.3.1 Função de geração em lote

Gerar um alerta por vez deixaria a GPU ociosa a maior parte do tempo. A função abaixo gera **em lote**, com dois cuidados:

- **Padding à esquerda.** As sequências do lote têm tamanhos diferentes e precisam ser completadas com padding. Num modelo que gera texto continuando a sequência, o padding vai à esquerda: se fosse à direita, o modelo teria que continuar depois dos tokens de padding.
- **`do_sample=False`.** A geração é determinística, sempre o token mais provável. Rodar de novo dá o mesmo resultado, e a comparação baseline × fine-tuned não depende de sorte.

In [ ]:
from tqdm.auto import tqdm

tokenizer.padding_side = "left"


def gerar_lote(conversas: list[list[dict]], tamanho_lote: int = 16) -> list[str]:
    saidas = []
    for i in tqdm(range(0, len(conversas), tamanho_lote)):
        inputs = tokenizer.apply_chat_template(
            conversas[i:i + tamanho_lote], add_generation_prompt=True, padding=True,
            return_dict=True, return_tensors="pt",
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        saidas += tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return saidas

### 6.3.2 Modelo ajustado

O modelo ajustado recebe o prompt do dataset da Fase 3: o `SYSTEM` curto e o alerta com o contexto, sem a política nem exemplos.

In [ ]:
saidas_ft = gerar_lote(teste_sft["prompt"])
print(saidas_ft[0])

### 6.3.3 Baseline

O baseline é o **mesmo prompt few-shot da Fase 1** (seção 1.3.2.6), com a política e os 3 exemplos resolvidos, rodando no modelo base. O prompt é copiado sem nenhuma mudança: melhorar o baseline aqui tornaria a comparação injusta a favor dele, e piorá-lo, a favor do fine-tuning.

Na Fase 1, o baseline foi medido numa amostra de 20 alertas. Aqui ele roda nos 221 alertas do teste, os mesmos do modelo ajustado. Os números podem mudar um pouco em relação à Fase 1, e os que valem para a decisão são os desta fase.

In [ ]:
POLITICA = """Política de severidade da organização:
- Sinal forte: ativo crítico = critica; ativo comum = alta; ativo de laboratório = media.
- Sinal moderado: ativo crítico = alta; ativo comum = media; ativo de laboratório = baixa.
- Sinal fraco e recorrente: ativo crítico = media; ativo comum = baixa; ativo de laboratório = baixa (falso positivo provável).
Responda APENAS com um JSON com as chaves: veredito (falso_positivo | verdadeiro_positivo), severidade (baixa | media | alta | critica) e justificativa."""

SYSTEM_BASELINE = f"Você é um analista de SOC.\n{POLITICA}"

FEW_SHOT = [
    (
        "EDR | Host: DC-01 | Usuário: svc_backup | Processo: powershell.exe -enc JAB... | Pai: winword.exe\n"
        "Ativo: controlador de domínio (criticidade: crítico) | Usuário: conta de serviço | Reputação: desconhecida | Ocorrências em 7 dias: 1",
        {"veredito": "verdadeiro_positivo", "severidade": "critica",
         "justificativa": "PowerShell codificado iniciado pelo Word em controlador de domínio, padrão de macro maliciosa no ativo mais sensível."},
    ),
    (
        "EDR | Host: WS-RH-14 | Usuário: m.souza | Processo: psexec.exe\n"
        "Ativo: estação de trabalho do RH (criticidade: comum) | Usuário: analista de RH, sem privilégio admin | Reputação: ferramenta legítima usada de forma suspeita | Ocorrências em 7 dias: 1",
        {"veredito": "verdadeiro_positivo", "severidade": "media",
         "justificativa": "Ferramenta administrativa incomum para o perfil do usuário, em estação não crítica."},
    ),
    (
        "IDS | Origem: LAB-SCAN-01 | Destino: LAB-WEB-03 | Evento: varredura de portas\n"
        "Ativo: servidor web de homologação (criticidade: laboratório) | Origem: scanner interno autorizado | Ocorrências em 7 dias: 120, nenhum incidente associado",
        {"veredito": "falso_positivo", "severidade": "baixa",
         "justificativa": "Varredura recorrente do scanner interno autorizado em ambiente de laboratório."},
    ),
]


def montar_mensagens(alerta: str, contexto: str) -> list[dict]:
    messages = [{"role": "system", "content": SYSTEM_BASELINE}]
    for entrada, saida in FEW_SHOT:
        messages.append({"role": "user", "content": entrada})
        messages.append({"role": "assistant", "content": json.dumps(saida, ensure_ascii=False)})
    messages.append({"role": "user", "content": f"{alerta}\n{contexto}"})
    return messages

Gerar com o adaptador desligado. O prompt few-shot é cerca de 4 vezes maior que o do modelo ajustado, então esta célula demora mais que a anterior:

In [ ]:
with model.disable_adapter():
    saidas_base = gerar_lote([montar_mensagens(ex["alerta"], ex["contexto"]) for ex in teste])
print(saidas_base[0])

## 6.4 Métricas

### 6.4.1 Validação estrita

A validação é **a mesma da Fase 1** (seção 1.3.2.8): a saída precisa ser um JSON com exatamente as 3 chaves e com categorias permitidas. Saída inválida conta como erro em todas as métricas, porque o sistema que consome a triagem (fila do SOC, SOAR) não consegue usá-la.

In [ ]:
VEREDITOS = {"falso_positivo", "verdadeiro_positivo"}
SEVERIDADES = {"baixa", "media", "alta", "critica"}
CHAVES = {"veredito", "severidade", "justificativa"}


def validar(saida: str) -> dict | None:
    try:
        d = json.loads(saida)
    except json.JSONDecodeError:
        return None
    if not isinstance(d, dict) or set(d) != CHAVES:
        return None
    if d["veredito"] not in VEREDITOS or d["severidade"] not in SEVERIDADES:
        return None
    return d


previstos = {
    "Baseline (few-shot)": [validar(s) for s in saidas_base],
    "Fine-tuned": [validar(s) for s in saidas_ft],
}

for nome, prev in previstos.items():
    print(f"{nome:20} inválidas: {sum(p is None for p in prev)} de {len(prev)}")

Montar uma tabela com o rótulo real e a previsão dos dois modelos para cada alerta. Uma saída inválida vira a categoria `invalida`, para aparecer nas tabelas como erro, e não sumir. A criticidade do ativo é extraída do `contexto`, como na seção 2.3.4:

In [ ]:
INVALIDA = "invalida"

resultados = pd.DataFrame({
    "template_id": teste["template_id"],
    "criticidade": pd.Series(teste["contexto"]).str.extract(r"criticidade: (\w+)", expand=False),
    "veredito_real": [r["veredito"] for r in reais],
    "severidade_real": [r["severidade"] for r in reais],
})
for nome, sufixo in [("Baseline (few-shot)", "base"), ("Fine-tuned", "ft")]:
    resultados[f"veredito_{sufixo}"] = [p["veredito"] if p else INVALIDA for p in previstos[nome]]
    resultados[f"severidade_{sufixo}"] = [p["severidade"] if p else INVALIDA for p in previstos[nome]]

resultados.head()

### 6.4.2 Comparar com as metas

Calcular as métricas da Fase 1 para os dois modelos. O **F1 macro de severidade** é a média do F1 das 4 severidades com o mesmo peso para cada uma. Ao contrário da acurácia, ele não fica alto só porque o modelo acerta as classes mais comuns (`alta` e `media`): errar todos os críticos derruba um quarto da média.

In [ ]:
from sklearn.metrics import f1_score

METAS = {
    "JSON válido": 0.98,
    "Acurácia de veredito": 0.85,
    "Acurácia de severidade": None,
    "F1 macro de severidade": None,
    "Recall de crítica": 0.90,
}


def calcular_metricas(sufixo: str) -> dict:
    r = resultados
    criticos = r[r["severidade_real"] == "critica"]
    return {
        "JSON válido": (r[f"severidade_{sufixo}"] != INVALIDA).mean(),
        "Acurácia de veredito": (r[f"veredito_{sufixo}"] == r["veredito_real"]).mean(),
        "Acurácia de severidade": (r[f"severidade_{sufixo}"] == r["severidade_real"]).mean(),
        "F1 macro de severidade": f1_score(
            r["severidade_real"], r[f"severidade_{sufixo}"], labels=ORDEM, average="macro", zero_division=0
        ),
        "Recall de crítica": (criticos[f"severidade_{sufixo}"] == "critica").mean(),
    }


metricas = {"Baseline (few-shot)": calcular_metricas("base"), "Fine-tuned": calcular_metricas("ft")}

tabela = pd.DataFrame(metricas)
tabela.insert(0, "Meta", pd.Series(METAS))
tabela["Atingiu?"] = [
    "—" if pd.isna(meta) else ("Sim" if ft >= meta else "Não")
    for meta, ft in zip(tabela["Meta"], tabela["Fine-tuned"])
]
print(f"Críticos no teste: {(resultados['severidade_real'] == 'critica').sum()}")
tabela.style.format("{:.0%}", subset=["Meta", "Baseline (few-shot)", "Fine-tuned"], na_rep="—")

**Leitura:**

- **JSON válido** mostra se o modelo aprendeu o **formato**. É o primeiro ganho esperado do fine-tuning, porque o formato é igual em todos os exemplos de treino.
- **Acurácia de veredito** e **recall de crítica** mostram se ele aprendeu a **política**: combinar o sinal técnico com a criticidade do ativo.
- A **acurácia de severidade** não tem meta na Fase 1, mas ajuda a ler o F1: se a acurácia é alta e o F1 macro é baixo, o modelo vai bem nas classes comuns e mal em alguma das raras.
- Com cerca de 18 críticos no teste, cada crítico errado custa cerca de 5 pontos de recall. A meta de 90% permite errar no máximo 1 ou 2.

### 6.4.3 Relatório por classe

O `classification_report` mostra precisão, recall e F1 de cada severidade. As saídas inválidas não formam uma classe, mas contam como erro no recall da classe real:

In [ ]:
from sklearn.metrics import classification_report

for nome, sufixo in [("Baseline (few-shot)", "base"), ("Fine-tuned", "ft")]:
    print(f"=== {nome} ===")
    print(classification_report(
        resultados["severidade_real"], resultados[f"severidade_{sufixo}"],
        labels=ORDEM, zero_division=0, digits=2,
    ))

**Leitura:** na linha `critica`, o **recall** é a métrica de negócio: dos alertas realmente críticos, quantos o modelo marcou como críticos. A **precisão** de `critica` mostra o outro lado: de tudo o que o modelo marcou como crítico, quanto era crítico de fato. Precisão baixa significa alarmes falsos no topo da fila, o que custa tempo do analista, mas não deixa ataques passarem.

### 6.4.4 Matriz de confusão de severidade

A matriz cruza a severidade real (linhas) com a prevista (colunas). A diagonal são os acertos. A **linha `critica` fora da diagonal**, destacada em vermelho, são os críticos classificados com severidade menor ou com saída inválida: os erros mais caros do projeto.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.metrics import confusion_matrix

COLUNAS = ORDEM + [INVALIDA]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (nome, sufixo) in zip(axes, [("Baseline (few-shot)", "base"), ("Fine-tuned", "ft")]):
    cm = confusion_matrix(resultados["severidade_real"], resultados[f"severidade_{sufixo}"], labels=COLUNAS)[:len(ORDEM)]
    ax.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    # linha "critica" (índice 0), da coluna "alta" até "invalida": crítico rebaixado ou perdido
    ax.add_patch(Rectangle((0.5, -0.5), len(COLUNAS) - 1, 1, fill=False, edgecolor="tab:red", linewidth=2.5))
    ax.set_xticks(range(len(COLUNAS)), COLUNAS)
    ax.set_yticks(range(len(ORDEM)), ORDEM)
    ax.set_xlabel("Prevista")
    ax.set_ylabel("Real")
    ax.set_title(nome)
fig.suptitle("Matriz de confusão de severidade no teste")
plt.tight_layout()
plt.show()

**Leitura:**

- Erros **logo ao lado da diagonal** (ex.: `alta` prevista como `media`) costumam ser a política aplicada com o sinal técnico certo, mas com a criticidade do ativo errada, ou o contrário.
- Erros **longe da diagonal** (ex.: `critica` prevista como `baixa`) indicam que o modelo não reconheceu o sinal técnico.
- A coluna `invalida` é o problema de formato. No baseline, ela concentra boa parte dos erros. No modelo ajustado, deveria estar praticamente vazia.

## 6.5 Custo do erro

Os erros de severidade não custam o mesmo. A política existe para ordenar a fila do L1, e o custo depende da direção do erro:

| Real \ Previsto | Severidade menor que a real | Severidade maior que a real |
|---|---|---|
| Crítico | ❌ **Custo altíssimo**: ataque real fica na fila | — |
| Baixo ou médio | — | Custo baixo: analista perde alguns minutos |

Por isso o recall de "crítica" pesa mais que a acurácia geral. Um modelo com acurácia de 90% que rebaixa metade dos críticos é pior, para o SOC, que um modelo com 80% de acurácia que acerta todos eles.

### 6.5.1 Contar os erros por direção

Classificar cada erro de severidade como **subestimado** (prevista menor que a real), **superestimado** (prevista maior) ou **inválido**:

In [ ]:
NIVEL = {"baixa": 0, "media": 1, "alta": 2, "critica": 3}


def direcao_do_erro(real: str, prevista: str) -> str:
    if prevista == INVALIDA:
        return "inválido"
    if NIVEL[prevista] < NIVEL[real]:
        return "subestimado"
    if NIVEL[prevista] > NIVEL[real]:
        return "superestimado"
    return "acerto"


custo = {}
for nome, sufixo in [("Baseline (few-shot)", "base"), ("Fine-tuned", "ft")]:
    resultados[f"erro_{sufixo}"] = [
        direcao_do_erro(r, p) for r, p in zip(resultados["severidade_real"], resultados[f"severidade_{sufixo}"])
    ]
    criticos = resultados[resultados["severidade_real"] == "critica"]
    custo[nome] = {
        **{k: int(v) for k, v in resultados[f"erro_{sufixo}"].value_counts().items()},
        "críticos perdidos (subestimados ou inválidos)": int((criticos[f"erro_{sufixo}"] != "acerto").sum()),
    }

pd.DataFrame(custo).reindex(
    ["acerto", "subestimado", "superestimado", "inválido", "críticos perdidos (subestimados ou inválidos)"]
).fillna(0).astype(int)

**Leitura:** num SOC, é melhor que os erros que restam sejam **superestimados**. Um modelo que, na dúvida, sobe a severidade gera trabalho extra para o L1. Um modelo que, na dúvida, desce a severidade esconde ataques.

### 6.5.2 Inspecionar os críticos perdidos

Listar os críticos que o modelo ajustado não marcou como críticos. Se eles se concentram num template ou numa criticidade, isso aponta a causa:

In [ ]:
perdidos = resultados[(resultados["severidade_real"] == "critica") & (resultados["severidade_ft"] != "critica")]

print(f"Críticos perdidos pelo fine-tuned: {len(perdidos)} de {(resultados['severidade_real'] == 'critica').sum()}")
for i in perdidos.index:
    print(f"\n--- {teste[i]['template_id']} | previsto: {resultados.at[i, 'severidade_ft']} ---")
    print(teste[i]["alerta"])
    print(teste[i]["contexto"])
    print(f"Saída: {saidas_ft[i][:200]}")

Perguntas para guiar a inspeção:

- O modelo errou a **criticidade do ativo** (tratou um servidor financeiro como estação comum)? Então ele não está lendo o contexto.
- Ele errou o **sinal técnico** (tratou PowerShell codificado como sinal moderado)? Então os templates de sinal forte do treino não cobrem esse tipo de alerta.
- Os erros se concentram num **único template**? Como a divisão é por template, um padrão de alerta muito diferente dos de treino pode ser o único responsável.

## 6.6 Justificativa coerente

As métricas automáticas só olham o veredito e a severidade. A justificativa é texto livre, e a meta da Fase 1 é um **checklist humano em 10 casos**: o analista só confia na triagem se entender por que o modelo chegou nela.

### 6.6.1 Sortear os casos

Sortear 10 casos do teste, garantindo que os críticos e os falsos positivos apareçam: 3 críticos, 3 falsos positivos e 4 dos demais. A semente fixa faz o sorteio ser o mesmo em toda execução:

In [ ]:
SEED = 42  # mesma semente da Fase 2

criticos = resultados[resultados["severidade_real"] == "critica"]
falsos_positivos = resultados[resultados["veredito_real"] == "falso_positivo"]
demais = resultados.drop(criticos.index.union(falsos_positivos.index))

casos = pd.concat([
    criticos.sample(3, random_state=SEED),
    falsos_positivos.sample(3, random_state=SEED),
    demais.sample(4, random_state=SEED),
]).index

for n, i in enumerate(casos, 1):
    print(f"=== Caso {n} ===")
    print(teste[i]["alerta"])
    print(teste[i]["contexto"])
    print(f"Esperado: {json.dumps(reais[i], ensure_ascii=False)}")
    print(f"Previsto: {saidas_ft[i]}\n")

### 6.6.2 Preencher o checklist

Para cada caso, responder com ✅ ou ❌:

- **Sinal técnico:** a justificativa cita o que foi detectado (processo, IOC, comportamento)?
- **Contexto:** ela cita o ativo, o usuário, a reputação ou a recorrência?
- **Coerente:** ela sustenta a severidade prevista? Ex.: uma justificativa que fala em "servidor crítico" com severidade `baixa` não é coerente.

O caso só conta como **coerente** se os três itens forem ✅.

| Caso | Sinal técnico | Contexto | Coerente com a severidade | Conta como coerente |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |
| 8 | | | | |
| 9 | | | | |
| 10 | | | | |
| **Total** | | | | **__ de 10** (meta: ≥ 8) |

> **Ponto de discussão:** uma justificativa pode ser coerente e a severidade estar errada, quando o modelo descreve bem os fatos mas aplica a política errado. Também pode acontecer o contrário: severidade certa com uma justificativa genérica, que poderia servir para qualquer alerta. O segundo caso é mais perigoso, porque parece correto e não ajuda o analista a conferir.

## 6.7 Salvar o resultado no Google Drive

Salvar as métricas no Drive, ao lado do adaptador. A demo da Fase 7 lê este arquivo para mostrar com que resultado o assistente foi medido:

In [ ]:
RESULTADO_PATH = f"{PROJETO_DIR}/avaliacao.json"

avaliacao = {
    "config_treino": config_treino,
    "metricas": {nome: {k: float(v) for k, v in m.items()} for nome, m in metricas.items()},
    "custo_do_erro": custo,
}
with open(RESULTADO_PATH, "w") as f:
    json.dump(avaliacao, f, ensure_ascii=False, indent=2)

print(f"Avaliação salva em {RESULTADO_PATH}")

## 6.8 Causa provável dos erros

O projeto **não tem iteração**: o adaptador da Fase 5 é a versão final. Mesmo assim, uma meta não atingida precisa vir com a causa provável, porque é ela que orienta uma próxima versão do assistente:

| Sintoma | Causa provável | Onde se resolveria |
|---|---|---|
| JSON inválido frequente | Máscara de loss ou template incorretos | Fase 3 |
| Recall de "crítica" baixo | Poucos críticos no treino ou templates de sinal forte que não cobrem o teste | Fase 2 |
| Treino bom e teste ruim | *Overfitting*: o modelo decorou os templates de treino | Fases 4 e 5 |
| Severidade certa, justificativa fraca | Capacidade do modelo 0.5B | Fase 1 (modelo `Qwen2.5-1.5B-Instruct`) |

Se mais de um sintoma aparece, o mais grave é o recall de "crítica", porque é o erro que deixa ataques passarem. A célula abaixo confere as metas automáticas em ordem de gravidade:

In [ ]:
ft = metricas["Fine-tuned"]

if ft["Recall de crítica"] < METAS["Recall de crítica"]:
    print("Sintoma mais grave: recall de 'crítica' baixo. Ver os críticos perdidos da seção 6.5.2.")
elif ft["JSON válido"] < METAS["JSON válido"]:
    print("Sintoma mais grave: JSON inválido frequente. Ver a coluna invalida da seção 6.4.4.")
elif ft["Acurácia de veredito"] < METAS["Acurácia de veredito"]:
    print("Sintoma mais grave: veredito fraco. Comparar com as curvas da seção 5.4.2 para separar overfitting de underfitting.")
else:
    print("Metas automáticas atingidas. A decisão final depende do checklist da justificativa (seção 6.6).")

## 6.9 Conclusão

Resultado da avaliação:

| Verificação | Resultado esperado | Onde conferir |
|---|---|---|
| Teste | 221 alertas de templates que o modelo nunca viu, alinhados nos dois datasets | Seção 6.2.2 |
| Comparação justa | Mesmo teste, mesma validação estrita e mesmo prompt few-shot da Fase 1 | Seções 6.3 e 6.4.1 |
| Metas automáticas | JSON válido ≥ 98%, veredito ≥ 85% e recall de "crítica" ≥ 90% | Seção 6.4.2 |
| Erros caros | Linha `critica` da matriz concentrada na diagonal e poucos subestimados | Seções 6.4.4 e 6.5 |
| Justificativa | Pelo menos 8 de 10 casos coerentes no checklist humano | Seção 6.6.2 |
| Causa dos erros | Sintoma mais grave identificado, sem retreino | Seção 6.8 |

**Conclusão obrigatória:** responder em até 5 linhas, com os números da tabela da seção 6.4.2: **o fine-tuning valeu a pena em relação ao baseline? Em quais métricas?** Se alguma meta não foi atingida, citar a causa provável da seção 6.8. Se não superou o baseline, dizer isso também é um resultado válido.

O adaptador avaliado aqui é a versão final, salva em `fine-tuning-soc/soc-triage-qwen2.5-0.5b-lora` no Drive. A Fase 7 carrega esse adaptador num runtime novo e o coloca atrás de uma demo no Gradio, com o mesmo `SYSTEM` e o mesmo chat template usados no treino e nesta avaliação.